# Test: silver fact load

Runs `src/setup`, `src/load_bronze_dims` and `src/load_silver_dims`, then runs `src/load_silver_facts` as a child notebook on small dummy rows written to
`silver.mandi_prices_clean`, and checks what lands in `silver.fact_mandi_prices`: the surrogate keys and prices,
that only the rows of the given run are loaded, reruns and updates, and the errors for a missing dimension key
and for a blank or unknown `run_id`.

Tests are plain pytest functions run inside this notebook with `ipytest`.
The notebook fails if any test fails, so it can be used as a job task.

Dummy rows use run ids and varieties starting with `pytest-`, so they never match real data.

Cleanup: after the tests, the dummy rows are deleted from clean and the fact table.
The tables and schemas that this run created are dropped; existing ones (and their data) are left as they were.

In [ ]:
%pip install pytest ipytest

## Parameters

In [ ]:
import uuid

RUN_PREFIX = f"pytest-{uuid.uuid4().hex[:8]}"

## State before setup

Recorded before `src/setup` runs, so cleanup only drops what this run creates.

In [ ]:
test_catalog = spark.catalog.currentCatalog()
CLEAN = f"{test_catalog}.silver.mandi_prices_clean"
FACT = f"{test_catalog}.silver.fact_mandi_prices"

# Tables and schemas that setup creates, tables first so they are dropped before their schema.
SETUP_TABLES = [
    FACT,
    CLEAN,
    f"{test_catalog}.silver.mandi_prices_rejected",
    f"{test_catalog}.silver.dim_market",
    f"{test_catalog}.silver.dim_commodity",
    f"{test_catalog}.silver.dim_date",
    f"{test_catalog}.bronze.raw_dim_market",
    f"{test_catalog}.bronze.raw_dim_commodity",
    f"{test_catalog}.bronze.raw_dim_date",
    f"{test_catalog}.bronze.raw_mandi_transactions",
]
SETUP_SCHEMAS = [f"{test_catalog}.silver", f"{test_catalog}.bronze"]

tables_existed = {name: spark.catalog.tableExists(name) for name in SETUP_TABLES}
schemas_existed = {
    name: len(spark.sql(f"SHOW SCHEMAS IN {test_catalog} LIKE '{name.split('.')[1]}'").collect()) == 1
    for name in SETUP_SCHEMAS
}

## Run setup and seed

In [ ]:
%run ../src/setup

In [ ]:
SEED_RUN_ID = f"{RUN_PREFIX}-seed"
dbutils.notebook.run("../src/load_bronze_dims", 600, {"run_id": SEED_RUN_ID})
dbutils.notebook.run("../src/load_silver_dims", 600, {"run_id": SEED_RUN_ID, "fail_on_error": "true"})

## pytest setup

In [ ]:
import sys

import ipytest
import pytest

# Keep pytest from writing __pycache__ into the Git folder.
sys.dont_write_bytecode = True

ipytest.autoconfig(raise_on_error=True, addopts=["-p", "no:cacheprovider"])

## Known dimension values

The first market and commodity of the seeded dimensions are used for the dummy rows.

In [ ]:
from datetime import date, datetime
from decimal import Decimal

_market = spark.table(f"{test_catalog}.silver.dim_market").orderBy("market_key").first()
_commodity = spark.table(f"{test_catalog}.silver.dim_commodity").orderBy("commodity_key").first()
MARKET_NAME = _market["market_name"]
MARKET_KEY = _market["market_key"]
DISTRICT_NAME = _market["district_name"]
COMMODITY_CODE = _commodity["commodity_code"]
COMMODITY_KEY = _commodity["commodity_key"]
COMMODITY_NAME = _commodity["commodity_name"]
DATE_KEY = 20260101

## Helpers

In [ ]:
def build__row(variety: str, **changes: object) -> dict[str, object]:
    """Build one valid dummy clean row.

    Args:
        variety: Variety of the row; use a different one per test so grains never collide.
        **changes: Clean columns to override, e.g. market="Unknown".

    Returns:
        Dict with every column of silver.mandi_prices_clean (run_id is set by insert__clean).
    """
    row = {
        "state": "Gujarat",
        "district": DISTRICT_NAME,
        "district_std": DISTRICT_NAME,
        "market": MARKET_NAME,
        "commodity_code": COMMODITY_CODE,
        "commodity": COMMODITY_NAME,
        "variety": f"{RUN_PREFIX}-{variety}",
        "grade": "FAQ",
        "arrival_date": date(2026, 1, 1),
        "min_price": Decimal("1000.00"),
        "max_price": Decimal("2000.00"),
        "modal_price": Decimal("1500.00"),
        "ingested_at": datetime.now(),
        "updated_at": datetime.now(),
    }
    row.update(changes)
    return row

In [ ]:
def insert__clean(run_id: str, rows: list[dict[str, object]]) -> None:
    """Append dummy rows to silver.mandi_prices_clean under a run id.

    Args:
        run_id: Run id of the rows.
        rows: Dicts from build__row.
    """
    schema = spark.table(CLEAN).schema
    data = [tuple({**row, "run_id": run_id}[f.name] for f in schema.fields) for row in rows]
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable(CLEAN)

In [ ]:
def run__fact(run_id: str) -> str:
    """Run src/load_silver_facts as a child notebook; raises if the notebook itself fails (blank widget, no catalog).

    Args:
        run_id: Value for the run_id widget.

    Returns:
        The notebook's exit value (empty, fact does not call exit).
    """
    return dbutils.notebook.run("../src/load_silver_facts", 600, {"run_id": run_id})

In [ ]:
def get__fact(variety_suffix: str) -> list["Row"]:
    """Return the fact rows of a dummy variety, as a list of Rows."""
    return spark.table(FACT).where(f"variety = '{RUN_PREFIX}-{variety_suffix}'").collect()

## Cleanup

In [ ]:
@pytest.fixture(scope="module", autouse=True)
def clean_up() -> "Generator[None, None, None]":
    """After the tests, delete the dummy rows; drop the tables/schemas only if this run created them.

    No CASCADE: dropping a schema fails instead of removing anything else in it.
    """
    yield

    for dim in ("market", "commodity", "date"):
        spark.sql(f"DELETE FROM {test_catalog}.bronze.raw_dim_{dim} WHERE _run_id = '{SEED_RUN_ID}'")
    spark.sql(f"DELETE FROM {FACT} WHERE variety LIKE '{RUN_PREFIX}%'")
    spark.sql(f"DELETE FROM {CLEAN} WHERE run_id LIKE '{RUN_PREFIX}%' OR variety LIKE '{RUN_PREFIX}%'")

    for name in SETUP_TABLES:
        if not tables_existed[name]:
            spark.sql(f"DROP TABLE IF EXISTS {name}")
    for name in SETUP_SCHEMAS:
        if not schemas_existed[name]:
            spark.sql(f"DROP SCHEMA IF EXISTS {name}")

## Test 1: keys and prices

In [ ]:
def test__fact_row_has_keys_and_prices() -> None:
    """A clean row becomes one fact row with the dimension keys, the degenerate columns and the prices."""
    run_id = f"{RUN_PREFIX}-keys"
    insert__clean(run_id, [build__row("keys", min_price=Decimal("1600.50"), modal_price=Decimal("1800.00"))])
    run__fact(run_id)

    rows = get__fact("keys")
    assert len(rows) == 1
    r = rows[0]
    assert (r["market_key"], r["commodity_key"], r["date_key"]) == (MARKET_KEY, COMMODITY_KEY, DATE_KEY)
    assert (r["variety"], r["grade"]) == (f"{RUN_PREFIX}-keys", "FAQ")
    assert (r["min_price"], r["max_price"], r["modal_price"]) == (Decimal("1600.50"), Decimal("2000.00"), Decimal("1800.00"))
    assert r["loaded_at"] is not None

## Test 2: only the rows of the run

In [ ]:
def test__only_rows_of_the_run_are_loaded() -> None:
    """Clean rows of another run are not loaded."""
    run_a = f"{RUN_PREFIX}-onlya"
    run_b = f"{RUN_PREFIX}-onlyb"
    insert__clean(run_a, [build__row("onlya")])
    insert__clean(run_b, [build__row("onlyb")])
    run__fact(run_a)

    assert len(get__fact("onlya")) == 1
    assert len(get__fact("onlyb")) == 0

## Test 3: reruns and updates

In [ ]:
def test__rerun_does_not_duplicate_rows() -> None:
    """Running the same run twice leaves one fact row per grain."""
    run_id = f"{RUN_PREFIX}-rerun"
    insert__clean(run_id, [build__row("rerun"), build__row("rerun", grade="Local")])
    run__fact(run_id)
    run__fact(run_id)

    assert len(get__fact("rerun")) == 2


def test__later_run_updates_the_fact_row() -> None:
    """When a later run updates a clean row, the fact row is updated and no second row is added."""
    first_run = f"{RUN_PREFIX}-update1"
    second_run = f"{RUN_PREFIX}-update2"
    insert__clean(first_run, [build__row("update", max_price=Decimal("2000.00"))])
    run__fact(first_run)

    spark.sql(f"""
        UPDATE {CLEAN}
        SET run_id = '{second_run}', max_price = 2500.00, modal_price = 1800.00
        WHERE variety = '{RUN_PREFIX}-update'
    """)
    run__fact(second_run)

    rows = get__fact("update")
    assert len(rows) == 1
    assert (rows[0]["max_price"], rows[0]["modal_price"]) == (Decimal("2500.00"), Decimal("1800.00"))

## Test 4: errors

In [ ]:
def test__missing_dimension_key_fails() -> None:
    """A clean row whose market is not in dim_market stops the load: none of the run's rows are loaded."""
    run_id = f"{RUN_PREFIX}-nokey"
    insert__clean(run_id, [build__row("nokey-ok"), build__row("nokey-bad", market="Pytest Unknown Market APMC")])

    run__fact(run_id)

    assert len(get__fact("nokey-ok")) == 0
    assert len(get__fact("nokey-bad")) == 0


def test__blank_run_id_fails() -> None:
    """A blank run_id raises."""
    with pytest.raises(Exception):
        run__fact("")


def test__run_id_without_clean_rows_fails() -> None:
    """A run_id with no rows in clean loads nothing."""
    run__fact(f"{RUN_PREFIX}-norows")

## Run

In [ ]:
ipytest.run("-v")